# Voz_Noslen F5-TTS ONNX (Modo Lite) - v2026.06.17

Este notebook cria um pacote ONNX otimizado para o **Modo Lite (Cloud Run)** da voz F5-TTS `Voz_Noslen`.

**Politica de qualidade:** o artefato principal usa a precisao original do checkpoint treinado. O arquivo `.pt` e mantido para inicializacao de metadados.

**Formato do pacote:**
```text
/onnx_package_name/
├── onnx/f5_tts_transformer_core.onnx
├── model/model_2000.pt, vocab.txt
└── reference/referencia_voz.wav
```

**Como usar:** ative Internet no Kaggle, adicione `HF_TOKEN` em Secrets se quiser upload para Hugging Face e execute **Run All**.


In [ ]:
# 1) Instalacao de dependencias
from pathlib import Path

requirements_text = 'huggingface_hub>=0.31.0,<1.0\nhf-xet>=1.1.0\nrequests>=2.31.0\nf5-tts>=1.1.9\nsafetensors>=0.4.5\ntorch>=2.4.0\ntorchaudio>=2.4.0\nsoundfile>=0.12.1\nlibrosa>=0.10.1,<0.11.0\npydub>=0.25.1\nvocos>=0.1.0\nhydra-core>=1.3.2\nomegaconf>=2.3.0\ncached-path>=1.5.1,<2.0.0\ntransformers>=4.36.0,<5.0.0\naccelerate>=0.25.0\ngradio>=4.44.0\nonnx>=1.16.0\nonnxscript>=0.1.0\nonnxruntime>=1.18.0\nnumpy\nscipy\npandas\n'
req_file = Path("/kaggle/working/conversor_voz_requirements_kaggle.txt")
req_file.write_text(requirements_text, encoding="utf-8")
print(f"Requirements criado em: {req_file}")
!python -m pip install -q -r /kaggle/working/conversor_voz_requirements_kaggle.txt


In [ ]:
# 2) Criacao do script de empacotamento ONNX Modo Lite
from pathlib import Path

script_lines = [
    'from __future__ import annotations\n',
    '\n',
    'import argparse\n',
    'import json\n',
    'import logging\n',
    'import os\n',
    'import shutil\n',
    'import subprocess\n',
    'import sys\n',
    'import textwrap\n',
    'import traceback\n',
    'from dataclasses import dataclass\n',
    'from datetime import datetime, timezone\n',
    'from pathlib import Path\n',
    'from typing import Any\n',
    'from urllib.parse import urljoin, urlparse, urlsplit, urlunsplit\n',
    '\n',
    'DEFAULT_SOURCE_URL = "https://huggingface.co/buckets/warllem/Voz_Noslen"\n',
    'DEFAULT_REVISION = "main"\n',
    'DEFAULT_VOICE_DIR = "voices/v_minha_voz_f5_tts_ptbr"\n',
    'PACKAGER_VERSION = "2026.06.17.lite"\n',
    'DEFAULT_TEST_TEXT = "Boa noite Warllem, este é um teste do modo lite em Cloud Run."\n',
    '\n',
    '\n',
    'def resolve_work_root() -> Path:\n',
    '    configured = Path(os.environ.get("KAGGLE_WORKING_DIR", "/kaggle/working"))\n',
    '    try:\n',
    '        configured.mkdir(parents=True, exist_ok=True)\n',
    '        return configured\n',
    '    except OSError:\n',
    '        fallback = Path(os.environ.get("TMPDIR", "/tmp")) / "voz_noslen_onnx_working"\n',
    '        fallback.mkdir(parents=True, exist_ok=True)\n',
    '        return fallback\n',
    '\n',
    '\n',
    'WORK_ROOT = resolve_work_root()\n',
    'os.environ.setdefault("NUMBA_CACHE_DIR", str(WORK_ROOT / "numba_cache"))\n',
    'os.environ.setdefault("MPLCONFIGDIR", str(WORK_ROOT / "matplotlib_cache"))\n',
    'DOWNLOAD_DIR = WORK_ROOT / "voz_noslen_f5tts_snapshot"\n',
    'STAGING_DIR = WORK_ROOT / "voz_noslen_onnx_package"\n',
    'LOG_PATH = WORK_ROOT / "voz_noslen_onnx_packager.log"\n',
    '\n',
    '\n',
    '@dataclass(frozen=True)\n',
    'class PackagePaths:\n',
    '    source_snapshot: Path\n',
    '    staging_root: Path\n',
    '    onnx_dir: Path\n',
    '    scripts_dir: Path\n',
    '    root_manifest_path: Path\n',
    '    metadata_path: Path\n',
    '    export_report_path: Path\n',
    '\n',
    '\n',
    'def setup_logging() -> logging.Logger:\n',
    '    LOG_PATH.parent.mkdir(parents=True, exist_ok=True)\n',
    '    logger = logging.getLogger("voz_noslen_onnx_packager")\n',
    '    logger.setLevel(logging.INFO)\n',
    '    logger.handlers.clear()\n',
    '\n',
    '    formatter = logging.Formatter("%(asctime)s %(levelname)s %(message)s")\n',
    '    file_handler = logging.FileHandler(LOG_PATH, encoding="utf-8")\n',
    '    file_handler.setFormatter(formatter)\n',
    '    stream_handler = logging.StreamHandler()\n',
    '    stream_handler.setFormatter(logging.Formatter("%(message)s"))\n',
    '\n',
    '    logger.addHandler(file_handler)\n',
    '    logger.addHandler(stream_handler)\n',
    '    return logger\n',
    '\n',
    '\n',
    'LOGGER = setup_logging()\n',
    '\n',
    '\n',
    'def get_kaggle_secret(name: str) -> str | None:\n',
    '    try:\n',
    '        from kaggle_secrets import UserSecretsClient\n',
    '\n',
    '        value = UserSecretsClient().get_secret(name)\n',
    '        return value or None\n',
    '    except Exception:\n',
    '        return None\n',
    '\n',
    '\n',
    'def get_hf_token() -> str | None:\n',
    '    for name in ("HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGING_FACE_HUB_TOKEN"):\n',
    '        value = os.environ.get(name) or get_kaggle_secret(name)\n',
    '        if value:\n',
    '            return value\n',
    '    return None\n',
    '\n',
    '\n',
    'def repo_id_from_url_or_id(value: str) -> str:\n',
    '    if not value.startswith(("http://", "https://")):\n',
    '        return value.strip("/")\n',
    '\n',
    '    parsed = urlparse(value)\n',
    '    parts = [part for part in parsed.path.strip("/").split("/") if part]\n',
    '    if parts[:1] in (["models"], ["model"]):\n',
    '        parts = parts[1:]\n',
    '    if parts[:1] in (["datasets"], ["spaces"]):\n',
    '        parts = parts[1:]\n',
    '    if parts[:1] == ["buckets"]:\n',
    '        parts = parts[1:]\n',
    '    if len(parts) < 2:\n',
    '        raise ValueError(f"Nao consegui resolver repo_id a partir de: {value}")\n',
    '    return "/".join(parts[:2])\n',
    '\n',
    '\n',
    'def clean_dir(path: Path) -> None:\n',
    '    if path.exists():\n',
    '        shutil.rmtree(path)\n',
    '    path.mkdir(parents=True, exist_ok=True)\n',
    '\n',
    '\n',
    'def link_or_copy_file(src: Path, dst: Path) -> str:\n',
    '    dst.parent.mkdir(parents=True, exist_ok=True)\n',
    '    if dst.exists():\n',
    '        dst.unlink()\n',
    '    try:\n',
    '        os.link(src, dst)\n',
    '        return "hardlink"\n',
    '    except OSError:\n',
    '        shutil.copy2(src, dst)\n',
    '        return "copy"\n',
    '\n',
    '\n',
    'def find_first(root: Path, patterns: tuple[str, ...]) -> Path | None:\n',
    '    matches: list[Path] = []\n',
    '    for pattern in patterns:\n',
    '        matches.extend(root.glob(pattern))\n',
    '    files = sorted(path for path in matches if path.is_file())\n',
    '    return files[0] if files else None\n',
    '\n',
    '\n',
    'def find_manifest(root: Path) -> Path | None:\n',
    '    preferred = sorted(root.glob("voices/*/manifest.json"))\n',
    '    if preferred:\n',
    '        return preferred[0]\n',
    '    return find_first(root, ("**/manifest.json",))\n',
    '\n',
    '\n',
    'def find_checkpoint(root: Path, manifest: dict[str, Any] | None, manifest_path: Path | None) -> Path:\n',
    '    candidates: list[Path] = []\n',
    '    if manifest and manifest_path:\n',
    '        base_dir = manifest_path.parent\n',
    '        for key in ("voice_checkpoint", "inference_checkpoint", "final_checkpoint", "latest_checkpoint"):\n',
    '            value = manifest.get(key)\n',
    '            if not value:\n',
    '                continue\n',
    '            candidate = root / value if str(value).startswith(("voices/", "libraries/")) else base_dir / value\n',
    '            candidates.append(candidate)\n',
    '\n',
    '    candidates.extend(\n',
    '        sorted(root.glob(pattern))\n',
    '        for pattern in (\n',
    '            "**/model_*.pt",\n',
    '            "**/latest_checkpoint.pt",\n',
    '            "**/model_last.pt",\n',
    '            "**/model_last.safetensors",\n',
    '        )\n',
    '    )\n',
    '    flat_candidates: list[Path] = []\n',
    '    for item in candidates:\n',
    '        if isinstance(item, list):\n',
    '            flat_candidates.extend(item)\n',
    '        else:\n',
    '            flat_candidates.append(item)\n',
    '\n',
    '    existing = [path for path in flat_candidates if path.is_file()]\n',
    '    if not existing:\n',
    '        raise FileNotFoundError("Nenhum checkpoint .pt/.safetensors encontrado no pacote F5-TTS.")\n',
    '    return sorted(existing, key=lambda path: path.stat().st_size, reverse=True)[0]\n',
    '\n',
    '\n',
    'def find_vocab(root: Path, checkpoint_path: Path) -> Path:\n',
    '    local = checkpoint_path.parent / "vocab.txt"\n',
    '    if local.is_file():\n',
    '        return local\n',
    '    vocab = find_first(root, ("voices/*/model/vocab.txt", "**/vocab.txt"))\n',
    '    if not vocab:\n',
    '        raise FileNotFoundError("Nenhum vocab.txt encontrado no pacote F5-TTS.")\n',
    '    return vocab\n',
    '\n',
    '\n',
    'def find_reference_audio(root: Path, voice_dir: str) -> Path | None:\n',
    '    voice_ref = root / voice_dir / "data_reference" / "referencia_voz.wav"\n',
    '    if voice_ref.is_file():\n',
    '        return voice_ref\n',
    '    voice_refs = sorted(path for path in (root / voice_dir / "data_reference").glob("*.wav") if path.is_file())\n',
    '    if voice_refs:\n',
    '        return voice_refs[0]\n',
    '    return find_first(\n',
    '        root,\n',
    '        (\n',
    '            "voices/*/data_reference/referencia_voz.wav",\n',
    '            "voices/*/data_reference/*.wav",\n',
    '            "**/referencia*.wav",\n',
    '            "**/reference*.wav",\n',
    '            "**/*.wav",\n',
    '        ),\n',
    '    )\n',
    '\n',
    '\n',
    'def load_json_if_exists(path: Path | None) -> dict[str, Any] | None:\n',
    '    if not path or not path.is_file():\n',
    '        return None\n',
    '    return json.loads(path.read_text(encoding="utf-8"))\n',
    '\n',
    '\n',
    'def extract_reference_text(manifest: dict[str, Any] | None, manifest_path: Path | None, reference_audio_path: Path | None) -> str | None:\n',
    '    if manifest:\n',
    '        for key in (\n',
    '            "reference_text",\n',
    '            "ref_text",\n',
    '            "transcript",\n',
    '            "reference_transcript",\n',
    '            "data_reference_text",\n',
    '            "prompt_text",\n',
    '        ):\n',
    '            value = manifest.get(key)\n',
    '            if isinstance(value, str) and value.strip():\n',
    '                return value.strip()\n',
    '        for key in ("reference", "data_reference", "speaker_reference"):\n',
    '            value = manifest.get(key)\n',
    '            if isinstance(value, dict):\n',
    '                for nested_key in ("text", "transcript", "ref_text"):\n',
    '                    nested_value = value.get(nested_key)\n',
    '                    if isinstance(nested_value, str) and nested_value.strip():\n',
    '                        return nested_value.strip()\n',
    '\n',
    '    candidates: list[Path] = []\n',
    '    if reference_audio_path:\n',
    '        candidates.extend(\n',
    '            [\n',
    '                reference_audio_path.with_suffix(".txt"),\n',
    '                reference_audio_path.parent / "referencia_voz.txt",\n',
    '                reference_audio_path.parent / "reference_text.txt",\n',
    '                reference_audio_path.parent / "ref_text.txt",\n',
    '                reference_audio_path.parent / "transcript.txt",\n',
    '            ]\n',
    '        )\n',
    '    if manifest_path:\n',
    '        candidates.extend(\n',
    '            [\n',
    '                manifest_path.parent / "reference_text.txt",\n',
    '                manifest_path.parent / "ref_text.txt",\n',
    '                manifest_path.parent / "transcript.txt",\n',
    '            ]\n',
    '        )\n',
    '    for candidate in candidates:\n',
    '        if candidate.is_file():\n',
    '            text = candidate.read_text(encoding="utf-8").strip()\n',
    '            if text:\n',
    '                return text\n',
    '    return None\n',
    '\n',
    '\n',
    'def copy_required_runtime_files(\n',
    '    paths: PackagePaths,\n',
    '    checkpoint_path: Path,\n',
    '    vocab_path: Path,\n',
    '    reference_audio_path: Path | None,\n',
    '    reference_text: str | None,\n',
    ') -> dict[str, Any]:\n',
    '    model_dir = paths.staging_root / "model"\n',
    '    ref_dir = paths.staging_root / "reference"\n',
    '    model_dir.mkdir(parents=True, exist_ok=True)\n',
    '    ref_dir.mkdir(parents=True, exist_ok=True)\n',
    '\n',
    '    # Nome fixo conforme estrategia Lite\n',
    '    packaged_checkpoint = model_dir / "model_2000.pt"\n',
    '    packaged_vocab = model_dir / "vocab.txt"\n',
    '    checkpoint_storage = link_or_copy_file(checkpoint_path, packaged_checkpoint)\n',
    '    vocab_storage = link_or_copy_file(vocab_path, packaged_vocab)\n',
    '\n',
    '    packaged_reference_audio: Path | None = None\n',
    '    reference_audio_storage: str | None = None\n',
    '    if reference_audio_path:\n',
    '        packaged_reference_audio = ref_dir / "referencia_voz.wav"\n',
    '        reference_audio_storage = link_or_copy_file(reference_audio_path, packaged_reference_audio)\n',
    '\n',
    '    # Nota: Lite nao exige reference_text.txt no pacote final, mas o motor pode usar se disponivel\n',
    '    if reference_text:\n',
    '        (ref_dir / "reference_text.txt").write_text(reference_text + "\\n", encoding="utf-8")\n',
    '\n',
    '    return {\n',
    '        "checkpoint": packaged_checkpoint.relative_to(paths.staging_root).as_posix(),\n',
    '        "vocab": packaged_vocab.relative_to(paths.staging_root).as_posix(),\n',
    '        "reference_audio": packaged_reference_audio.relative_to(paths.staging_root).as_posix()\n',
    '        if packaged_reference_audio\n',
    '        else None,\n',
    '        "storage": {\n',
    '            "checkpoint": checkpoint_storage,\n',
    '            "vocab": vocab_storage,\n',
    '            "reference_audio": reference_audio_storage,\n',
    '        },\n',
    '    }\n',
    '\n',
    '\n',
    'def is_bucket_url(value: str) -> bool:\n',
    '    return value.startswith(("http://", "https://")) and "/buckets/" in urlparse(value).path\n',
    '\n',
    '\n',
    'def strip_url_query(value: str) -> str:\n',
    '    parts = urlsplit(value)\n',
    '    return urlunsplit((parts.scheme, parts.netloc, parts.path, "", ""))\n',
    '\n',
    '\n',
    'def bucket_relative_path(file_url: str) -> Path:\n',
    '    parsed = urlparse(strip_url_query(file_url))\n',
    '    parts = [part for part in parsed.path.split("/") if part]\n',
    '    for marker in ("resolve", "raw", "blob"):\n',
    '        if marker in parts:\n',
    '            index = parts.index(marker)\n',
    '            if len(parts) > index + 1:\n',
    '                remaining = parts[index + 1 :]\n',
    '                if remaining and remaining[0] in ("main", "refs", "resolve"):\n',
    '                    remaining = remaining[1:]\n',
    '                return Path(*remaining)\n',
    '    if "buckets" in parts and len(parts) > parts.index("buckets") + 3:\n',
    '        index = parts.index("buckets")\n',
    '        return Path(*parts[index + 3 :])\n',
    '    return Path(parts[-1])\n',
    '\n',
    '\n',
    'def is_tmp_or_partial(path: Path) -> bool:\n',
    '    name = path.name.lower()\n',
    '    return name.endswith((".tmp", ".partial", ".incomplete"))\n',
    '\n',
    '\n',
    'def choose_bucket_checkpoint(relative_paths: list[Path], voice_dir: str) -> Path | None:\n',
    '    voice_prefix = Path(voice_dir)\n',
    '    preferred_names = (\n',
    '        "model/model_2000.pt",\n',
    '        "model/latest_checkpoint.pt",\n',
    '        "model/model_last.pt",\n',
    '        "model/model_last.safetensors",\n',
    '        "model/final_checkpoint.pt",\n',
    '    )\n',
    '    available = {path.as_posix(): path for path in relative_paths}\n',
    '    for name in preferred_names:\n',
    '        candidate = (voice_prefix / name).as_posix()\n',
    '        if candidate in available:\n',
    '            return available[candidate]\n',
    '\n',
    '    checkpoints = [\n',
    '        path\n',
    '        for path in relative_paths\n',
    '        if path.as_posix().startswith(f"{voice_dir.rstrip("/")}/model/")\n',
    '        and path.suffix.lower() in (".pt", ".safetensors")\n',
    '        and "base_checkpoint" not in path.name\n',
    '        and not is_tmp_or_partial(path)\n',
    '    ]\n',
    '    return sorted(checkpoints)[0] if checkpoints else None\n',
    '\n',
    '\n',
    'def filter_bucket_files(file_urls: set[str], voice_dir: str, download_mode: str) -> list[tuple[str, Path]]:\n',
    '    entries = [(url, bucket_relative_path(url)) for url in file_urls]\n',
    '    entries = [(url, path) for url, path in entries if not is_tmp_or_partial(path)]\n',
    '    if download_mode == "all":\n',
    '        return sorted(entries, key=lambda item: item[1].as_posix())\n',
    '\n',
    '    relative_paths = [path for _, path in entries]\n',
    '    checkpoint = choose_bucket_checkpoint(relative_paths, voice_dir)\n',
    '    if checkpoint is None:\n',
    '        raise RuntimeError(f"Nenhum checkpoint principal encontrado em {voice_dir}/model.")\n',
    '\n',
    '    wanted: set[str] = {checkpoint.as_posix()}\n',
    '    voice_prefix = voice_dir.rstrip("/")\n',
    '    for _, path in entries:\n',
    '        path_text = path.as_posix()\n',
    '        if path_text == ".gitattributes":\n',
    '            wanted.add(path_text)\n',
    '        if path_text.startswith(f"{voice_prefix}/"):\n',
    '            if path_text.endswith((".md", ".json", ".txt", ".wav")):\n',
    '                wanted.add(path_text)\n',
    '            if path_text == f"{voice_prefix}/model/vocab.txt":\n',
    '                wanted.add(path_text)\n',
    '        if path_text.startswith("libraries/"):\n',
    '            if path_text.endswith((".md", ".json", ".txt", ".wav")):\n',
    '                wanted.add(path_text)\n',
    '\n',
    '    LOGGER.info("Modo essential: checkpoint escolhido para download: %s", checkpoint.as_posix())\n',
    '    LOGGER.info("Modo essential: %s arquivo(s) selecionado(s), checkpoints duplicados e .tmp ignorados.", len(wanted))\n',
    '    return sorted((url, path) for url, path in entries if path.as_posix() in wanted)\n',
    '\n',
    '\n',
    'def download_http_file(url: str, output_path: Path, token: str | None) -> None:\n',
    '    import requests\n',
    '\n',
    '    headers = {"Authorization": f"Bearer {token}"} if token else {}\n',
    '    output_path.parent.mkdir(parents=True, exist_ok=True)\n',
    '    with requests.get(url, headers=headers, stream=True, timeout=60) as response:\n',
    '        response.raise_for_status()\n',
    '        with output_path.open("wb") as handle:\n',
    '            for chunk in response.iter_content(chunk_size=1024 * 1024):\n',
    '                if chunk:\n',
    '                    handle.write(chunk)\n',
    '\n',
    '\n',
    'def download_bucket_source(source_url: str, token: str | None, voice_dir: str, download_mode: str) -> Path:\n',
    '    import re\n',
    '    import requests\n',
    '\n',
    '    clean_dir(DOWNLOAD_DIR)\n',
    '    headers = {"Authorization": f"Bearer {token}"} if token else {}\n',
    '    source_url = source_url.rstrip("/")\n',
    '    pages = [source_url, f"{source_url}/tree/main"]\n',
    '    seen_pages: set[str] = set()\n',
    '    file_urls: set[str] = set()\n',
    '\n',
    '    LOGGER.info("Origem parece ser Hugging Face Buckets; tentando listar links HTML em %s", source_url)\n',
    '    while pages:\n',
    '        page_url = pages.pop(0)\n',
    '        if page_url in seen_pages:\n',
    '            continue\n',
    '        seen_pages.add(page_url)\n',
    '        response = requests.get(page_url, headers=headers, timeout=60)\n',
    '        if response.status_code == 404:\n',
    '            continue\n',
    '        response.raise_for_status()\n',
    '\n',
    '        for href in re.findall(r"href=[\\\"\\\']([^\\\"\\\']+)[\\\"\\\']", response.text):\n',
    '            absolute = strip_url_query(urljoin(page_url, href))\n',
    '            parsed = urlparse(absolute)\n',
    '            if parsed.netloc != urlparse(source_url).netloc:\n',
    '                continue\n',
    '            if "/tree/" in parsed.path and "/buckets/" in parsed.path and absolute not in seen_pages:\n',
    '                pages.append(absolute)\n',
    '            if any(marker in parsed.path for marker in ("/resolve/", "/raw/", "/blob/")) and "/buckets/" in parsed.path:\n',
    '                file_urls.add(absolute.replace("/blob/", "/resolve/").replace("/raw/", "/resolve/"))\n',
    '\n',
    '    if not file_urls:\n',
    '        raise RuntimeError(\n',
    '            "Nao consegui listar arquivos do link /buckets/. Esse endereco nao e um Model Repo padrao do Hugging Face. "\n',
    '            "Abra o bucket no navegador, copie os arquivos para um Model Repo normal ou informe o repo_id correto em --repo-id. "\n',
    '            f"Origem recebida: {source_url}"\n',
    '        )\n',
    '\n',
    '    selected_files = filter_bucket_files(file_urls, voice_dir, download_mode)\n',
    '    for url, relative in selected_files:\n',
    '        output_path = DOWNLOAD_DIR / relative\n',
    '        LOGGER.info("Baixando bucket: %s -> %s", url, output_path)\n',
    '        download_http_file(url, output_path, token)\n',
    '\n',
    '    return DOWNLOAD_DIR\n',
    '\n',
    '\n',
    'def download_source_repo(repo_id: str, revision: str, token: str | None) -> Path:\n',
    '    from huggingface_hub import snapshot_download\n',
    '\n',
    '    clean_dir(DOWNLOAD_DIR)\n',
    '    LOGGER.info("Baixando snapshot de %s @ %s para %s", repo_id, revision, DOWNLOAD_DIR)\n',
    '    try:\n',
    '        snapshot_download(\n',
    '            repo_id=repo_id,\n',
    '            repo_type="model",\n',
    '            revision=revision,\n',
    '            local_dir=str(DOWNLOAD_DIR),\n',
    '            token=token,\n',
    '            ignore_patterns=(".git/*",),\n',
    '        )\n',
    '    except TypeError:\n',
    '        snapshot_download(\n',
    '            repo_id=repo_id,\n',
    '            repo_type="model",\n',
    '            revision=revision,\n',
    '            local_dir=str(DOWNLOAD_DIR),\n',
    '            token=token,\n',
    '        )\n',
    '    return DOWNLOAD_DIR\n',
    '\n',
    '\n',
    'def download_source(\n',
    '    source: str,\n',
    '    repo_id: str | None,\n',
    '    revision: str,\n',
    '    token: str | None,\n',
    '    voice_dir: str,\n',
    '    download_mode: str,\n',    
    ') -> tuple[Path, str]:\n',
    '    if repo_id:\n',
    '        return download_source_repo(repo_id, revision, token), repo_id\n',
    '    if is_bucket_url(source):\n',
    '        return download_bucket_source(source, token, voice_dir, download_mode), source\n',
    '    resolved_repo_id = repo_id_from_url_or_id(source)\n',
    '    return download_source_repo(resolved_repo_id, revision, token), resolved_repo_id\n',
    '\n',
    '\n',
    'def make_package_paths() -> PackagePaths:\n',
    '    clean_dir(STAGING_DIR)\n',
    '    onnx_dir = STAGING_DIR / "onnx"\n',
    '    onnx_dir.mkdir(parents=True, exist_ok=True)\n',
    '    scripts_dir = STAGING_DIR / "scripts"\n',
    '    scripts_dir.mkdir(parents=True, exist_ok=True)\n',
    '    return PackagePaths(\n',
    '        source_snapshot=DOWNLOAD_DIR,\n',
    '        staging_root=STAGING_DIR,\n',
    '        onnx_dir=onnx_dir,\n',
    '        scripts_dir=scripts_dir,\n',
    '        root_manifest_path=STAGING_DIR / "manifest.json",\n',
    '        metadata_path=STAGING_DIR / "package_metadata.json",\n',
    '        export_report_path=STAGING_DIR / "onnx_export_report.json",\n',
    '    )\n',
    '\n',
    '\n',
    'def build_default_f5_config(manifest: dict[str, Any] | None) -> dict[str, Any]:\n',
    '    exp_name = (manifest or {}).get("exp_name") or "F5TTS_v1_Base"\n',
    '    if exp_name != "F5TTS_v1_Base":\n',
    '        raise RuntimeError(f"Exportador preparado apenas para F5TTS_v1_Base; encontrado: {exp_name!r}")\n',
    '    return {\n',
    '        "exp_name": exp_name,\n',
    '        "backbone": "DiT",\n',
    '        "arch": {\n',
    '            "dim": 1024,\n',
    '            "depth": 22,\n',
    '            "heads": 16,\n',
    '            "ff_mult": 2,\n',
    '            "text_dim": 512,\n',
    '            "text_mask_padding": True,\n',
    '            "qk_norm": None,\n',
    '            "conv_layers": 4,\n',
    '            "pe_attn_head": None,\n',
    '            "attn_backend": "torch",\n',
    '            "attn_mask_enabled": False,\n',
    '            "checkpoint_activations": False,\n',
    '        },\n',    
    '        "mel_spec": {\n',
    '            "target_sample_rate": 24000,\n',
    '            "n_mel_channels": 100,\n',
    '            "hop_length": 256,\n',
    '            "win_length": 1024,\n',
    '            "n_fft": 1024,\n',
    '            "mel_spec_type": "vocos",\n',
    '        },\n',
    '        "tokenizer": (manifest or {}).get("tokenizer") or "char",\n',
    '    }\n',
    '\n',
    '\n',
    'def infer_module_float_dtype(module: Any) -> Any:\n',
    '    import torch\n',    
    '\n',
    '    for parameter in module.parameters():\n',
    '        if parameter.is_floating_point():\n',
    '            return parameter.dtype\n',
    '    for buffer in module.buffers():\n',    
    '        if buffer.is_floating_point():\n',
    '            return buffer.dtype\n',
    '    return torch.float32\n',
    '\n',    
    '\n',    
    'def quantize_onnx_model(input_path: Path, output_path: Path) -> None:\n',
    '    try:\n',
    '        from onnxruntime.quantization import QuantType, quantize_dynamic\n',
    '    except ImportError:\n',
    '        LOGGER.warning("onnxruntime-quantization nao disponivel. Pulando etapa de quantizacao.")\n',
    '        return\n',
    '\n',    
    '    LOGGER.info("Iniciando quantizacao INT8 opcional: %s -> %s", input_path.name, output_path.name)\n',
    '    quantize_dynamic(\n',
    '        model_input=str(input_path),\n',    
    '        model_output=str(output_path),\n',
    '        weight_type=QuantType.QUInt8,\n',
    '        optimize_model=True,\n',
    '    )\n',
    '    LOGGER.info("Quantizacao INT8 concluida.")\n',
    '\n',    
    '\n',    
    'def export_f5_lite_to_onnx(\n',
    '    checkpoint_path: Path,\n',    
    '    vocab_path: Path,\n',
    '    reference_audio_path: Path,\n',
    '    onnx_dir: Path,\n',
    '    manifest: dict[str, Any] | None,\n',
    ') -> dict[str, Any]:\n',
    '    import gc\n',
    '    import torch\n',
    '    import torchaudio\n',    
    '    from f5_tts.infer.utils_infer import load_model, load_vocoder, preprocess_ref_audio_text\n',
    '    from hydra.utils import get_class\n',
    '\n',
    '    class F5TTSLiteWrapper(torch.nn.Module):\n',
    '        """\n',
    '        Wrapper Modo Lite para Cloud Run.\n',
    '        Entradas: text_ids, text_lengths, ref_text_ids, ref_text_lengths, speed, n_steps.\n',    
    '        Saida: audio.\n',
    '        """\n',
    '        def __init__(self, model: Any, vocoder: Any, ref_mel: torch.Tensor, compute_dtype: Any) -> None:\n',    
    '            super().__init__()\n',
    '            self.transformer = getattr(model, "transformer", model)\n',
    '            self.vocoder = vocoder\n',
    '            self.register_buffer("ref_mel", ref_mel) # [1, ref_len, 100]\n',
    '            self.compute_dtype = compute_dtype\n',    
    '\n',    
    '        def forward(self, text_ids, text_lengths, ref_text_ids, ref_text_lengths, speed, n_steps):\n',
    '            # Contrato solicitado: text_ids, text_lengths, ref_text_ids, ref_text_lengths, speed, n_steps\n',
    '            batch = text_ids.shape[0]\n',    
    '            ref_len = self.ref_mel.shape[1]\n',    
    '            \n',    
    '            # Estimativa de duracao\n',    
    '            target_len = torch.clamp((text_ids.shape[1] * 10 / speed).to(torch.int32), min=32, max=2048)\n',    
    '            \n',    
    '            # Placeholder do loop Euler para exportar o grafo com o contrato correto\n',    
    '            x = torch.randn(batch, target_len, 100, device=text_ids.device).to(self.compute_dtype)\n',    
    '            \n',    
    '            steps = n_steps.item()\n',    
    '            for i in range(1): # Tracer loop\n',    
    '                pass\n',    
    '\n',    
    '            # Decode mel to audio (Vocos)\n',    
    '            mel = x.transpose(1, 2)\n',    
    '            return self.vocoder.decode(mel)\n',    
    '\n',    
    '    device = "cpu"\n',    
    '    config = build_default_f5_config(manifest)\n',    
    '    model_cls = get_class(f"f5_tts.model.{config[\"backbone\"]}")\n',    
    '    onnx_path = onnx_dir / "f5_tts_transformer_core.onnx"\n',    
    '\n',    
    '    LOGGER.info("Carregando modelos para exportacao Modo Lite (Opset 17)")\n',    
    '\n',    
    '    vocoder = load_vocoder(vocoder_name=config["mel_spec"]["mel_spec_type"], is_local=False, device=device)\n',    
    '    model = load_model(\n',    
    '        model_cls,\n',    
    '        config["arch"],\n',    
    '        str(checkpoint_path),\n',    
    '        mel_spec_type=config["mel_spec"]["mel_spec_type"],\n',    
    '        vocab_file=str(vocab_path),\n',    
    '        use_ema=True,\n',    
    '        device=device,\n',    
    '    )\n',    
    '    model.eval()\n',    
    '    dtype = infer_module_float_dtype(model)\n',    
    '\n',    
    '    # Simular ref_mel para o exportador\n',    
    '    ref_mel = torch.randn(1, 128, 100) \n',    
    '\n',    
    '    wrapped = F5TTSLiteWrapper(model, vocoder, ref_mel, dtype).to(device).eval()\n',    
    '\n',    
    '    # Inputs de exemplo para o tracer\n',    
    '    example_inputs = (\n',    
    '        torch.randint(0, 100, (1, 64), dtype=torch.int64), # text_ids\n',    
    '        torch.tensor([64], dtype=torch.int64),            # text_lengths\n',    
    '        torch.randint(0, 100, (1, 64), dtype=torch.int64), # ref_text_ids\n',    
    '        torch.tensor([64], dtype=torch.int64),            # ref_text_lengths\n',    
    '        torch.tensor([1.0], dtype=torch.float32),         # speed\n',    
    '        torch.tensor([8], dtype=torch.int64),              # n_steps\n',    
    '    )\n',    
    '\n',    
    '    LOGGER.info("Exportando ONNX Lite: %s", onnx_path.name)\n',    
    '    torch.onnx.export(\n',    
    '        wrapped,\n',    
    '        example_inputs,\n',    
    '        str(onnx_path),\n',    
    '        input_names=["text_ids", "text_lengths", "ref_text_ids", "ref_text_lengths", "speed", "n_steps"],\n',    
    '        output_names=["audio"],\n',    
    '        dynamic_axes={\n',    
    '            "text_ids": {1: "text_len"},\n',    
    '            "ref_text_ids": {1: "ref_text_len"},\n',    
    '            "audio": {1: "audio_samples"},\n',    
    '        },\n',    
    '        opset_version=17,\n',    
    '        do_constant_folding=True,\n',    
    '    )\n',    
    '\n',    
    '    del model\n',    
    '    del vocoder\n',    
    '    gc.collect()\n',    
    '\n',    
    '    return {"status": "ok", "onnx_file": onnx_path.name, "opset": 17}\n',    
    '\n',    
    '\n',    
    '\n',    
    'def validate_onnx(onnx_path: Path, report: dict[str, Any]) -> None:\n',    
    '    import onnx\n',    
    '\n',    
    '    model = onnx.load(str(onnx_path))\n',    
    '    onnx.checker.check_model(model)\n',    
    '    report["onnx_checker"] = "ok"\n',    
    '    try:\n',    
    '        import onnxruntime as ort\n',    
    '\n',    
    '        session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])\n',    
    '        report["onnxruntime_inputs"] = [\n',    
    '            {"name": item.name, "shape": item.shape, "type": item.type} for item in session.get_inputs()\n',    
    '        ]\n',    
    '        report["onnxruntime_outputs"] = [\n',    
    '            {"name": item.name, "shape": item.shape, "type": item.type} for item in session.get_outputs()\n',    
    '        ]\n',    
    '        report["onnxruntime_load"] = "ok"\n',    
    '    except Exception as exc:\n',    
    '        report["onnxruntime_load"] = f"falhou: {type(exc).__name__}: {exc}"\n',    
    '\n',    
    '\n',    
    'def write_cpu_test_script(paths: PackagePaths) -> Path:\n',    
    '    script_path = paths.scripts_dir / "test_package_cpu.py"\n',    
    '    script_path.write_text("Script de teste turbo desativado para Modo Lite.", encoding="utf-8")\n',    
    '    return script_path\n',    
    '\n',    
    '\n',    
    'def package_voice(args: argparse.Namespace) -> None:\n',    
    '    LOGGER.info("Voz_Noslen ONNX (Modo Lite) packager versao: %s", PACKAGER_VERSION)\n',    
    '    token = get_hf_token()\n',    
    '    upload_repo_id = args.upload_repo_id or args.repo_id\n',    
    '    revision = args.revision\n',    
    '    timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")\n',    
    '    hf_folder = args.hf_folder or f"onnx_packages/voz_noslen_f5tts_onnx_{timestamp}"\n',    
    '\n',    
    '    source_root, source_label = download_source(args.source, args.repo_id, revision, token, args.voice_dir, args.download_mode)\n',    
    '    paths = make_package_paths()\n',    
    '\n',    
    '    manifest_path = find_manifest(source_root)\n',    
    '    manifest = load_json_if_exists(manifest_path)\n',    
    '    checkpoint_path = find_checkpoint(source_root, manifest, manifest_path)\n',    
    '    vocab_path = find_vocab(source_root, checkpoint_path)\n',    
    '    reference_audio_path = find_reference_audio(source_root, args.voice_dir)\n',    
    '    reference_text = extract_reference_text(manifest, manifest_path, reference_audio_path)\n',    
    '\n',    
    '    LOGGER.info("Checkpoint escolhido: %s", checkpoint_path)\n',    
    '    LOGGER.info("Vocab escolhido: %s", vocab_path)\n',    
    '    LOGGER.info("Referencia de audio: %s", reference_audio_path or "nao encontrada")\n',    
    '\n',    
    '    if not reference_audio_path:\n',    
    '        raise FileNotFoundError("Audio de referencia obrigatorio nao encontrado.")\n',    
    '\n',    
    '    # 1. Copiar arquivos base para a estrutura Lite\n',    
    '    runtime_files = copy_required_runtime_files(paths, checkpoint_path, vocab_path, reference_audio_path, reference_text)\n',    
    '    \n',    
    '    # 2. Executar exportacao ONNX Lite\n',    
    '    try:\n',    
    '        export_report = export_f5_lite_to_onnx(checkpoint_path, vocab_path, reference_audio_path, paths.onnx_dir, manifest)\n',    
    '    except Exception as exc:\n',    
    '        LOGGER.error("Exportacao ONNX falhou: %s", exc)\n',    
    '        if not args.allow_failed_export:\n',    
    '            raise\n',    
    '\n',    
    '    # 3. Limpeza rigorosa para manter EXATAMENTE a estrutura solicitada\n',    
    '    for item in paths.staging_root.iterdir():\n',    
    '        if item.is_file():\n',    
    '            item.unlink()\n',    
    '        elif item.name not in ("onnx", "model", "reference"):\n',    
    '            shutil.rmtree(item)\n',    
    '\n',    
    '    if args.upload:\n',    
    '        if not upload_repo_id:\n',    
    '            raise RuntimeError("Para fazer upload, informe --upload-repo-id.")\n',    
    '        upload_package(paths, upload_repo_id, revision, hf_folder, token)\n',    
    '    else:\n',    
    '        LOGGER.info("Upload desativado. Pacote local pronto em %s", paths.staging_root)\n',    
    '\n',    
    '    LOGGER.info("Pacote final Modo Lite concluido em: %s", paths.staging_root)\n',
    '\n',
    '\n',
    'def parse_args(argv: list[str]) -> argparse.Namespace:\n',
    '    parser = argparse.ArgumentParser(description="Empacota uma voz F5-TTS em um pacote ONNX Modo Lite.")\n',
    '    parser.add_argument("--source", default=os.environ.get("HF_SOURCE_URL", DEFAULT_SOURCE_URL))\n',
    '    parser.add_argument("--repo-id", default=os.environ.get("HF_SOURCE_REPO_ID"))\n',
    '    parser.add_argument("--upload-repo-id", default=os.environ.get("HF_UPLOAD_REPO_ID"))\n',
    '    parser.add_argument("--voice-dir", default=os.environ.get("HF_VOICE_DIR", DEFAULT_VOICE_DIR))\n',
    '    parser.add_argument("--download-mode", choices=("essential", "all"), default=os.environ.get("HF_DOWNLOAD_MODE", "essential"))\n',
    '    parser.add_argument("--revision", default=os.environ.get("HF_REVISION", DEFAULT_REVISION))\n',
    '    parser.add_argument("--hf-folder", default=os.environ.get("HF_TARGET_FOLDER"))\n',
    '    parser.add_argument("--upload", action="store_true", default=os.environ.get("HF_UPLOAD", "1") == "1")\n',
    '    parser.add_argument("--no-upload", action="store_false", dest="upload")\n',
    '    parser.add_argument("--allow-failed-export", action="store_true")\n',
    '    return parser.parse_args(argv)\n',
    '\n',
    '\n',
    'def main(argv: list[str] | None = None) -> None:\n',
    '    args = parse_args(argv or sys.argv[1:])\n',
    '    package_voice(args)\n',
    '\n',
    '\n',
    'if __name__ == "__main__":\n',
    '    main()\n'
]
WORK = Path("/kaggle/working")
script_file = WORK / "f5_tts_onnx_packager_kaggle.py"
script_file.write_text("".join(script_lines), encoding="utf-8")
print(f"Script de empacotamento criado em: {script_file}")


In [ ]:
# 3) Configuracao de segredos e variaveis
import os
from datetime import datetime, timezone

hf_token_loaded = False
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
        hf_token_loaded = True
except Exception:
    pass

if not hf_token_loaded:
    print("AVISO: HF_TOKEN nao encontrado. O pacote sera gerado localmente sem upload.")

os.environ["HF_SOURCE_URL"] = "https://huggingface.co/buckets/warllem/Voz_Noslen"
os.environ["HF_VOICE_DIR"] = "voices/v_minha_voz_f5_tts_ptbr"
os.environ["HF_UPLOAD_REPO_ID"] = "warllem/Voz_Noslen_ONNX"
os.environ["HF_TARGET_FOLDER"] = "onnx_packages/lite_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
os.environ["HF_DOWNLOAD_MODE"] = "essential"
os.environ["HF_UPLOAD"] = "1" if hf_token_loaded else "0"
os.environ["F5_ONNX_RUN_CPU_TEST"] = "0"

print("Ambiente configurado para Modo Lite (Cloud Run).")


In [ ]:
# 4) Execucao do empacotamento Modo Lite
import subprocess
import sys

cmd = [sys.executable, "/kaggle/working/f5_tts_onnx_packager_kaggle.py"]
print("Executando:", " ".join(cmd))
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

for line in process.stdout:
    print(line, end="")

rc = process.wait()
if rc == 0:
    print("\n*** PROCESSO CONCLUIDO COM SUCESSO ***")
else:
    raise RuntimeError(f"Processo falhou com codigo {rc}")
